# 06 · Mixed precision: FP16, BF16 and loss scaling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/distributed/06_mixed_precision.ipynb) &nbsp; 📖 Concept deck: [Mixed precision and FP8 training](https://nilesh-salpe.github.io/visual-ai/distributed/09-mixed-precision-fp8/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** BF16 rounding, a dynamic loss scaler, and a training loop with FP16 gradients and FP32 master weights that works only when the loss is scaled.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

r = np.random.default_rng(0)
Dm = 8
Xs = r.normal(0, 1e-4, (64, Dm)); w_star = r.normal(size=Dm); ys = Xs @ w_star     # tiny features → tiny gradients

def loss_and_grad(w):
    e = Xs @ w - ys
    return float(np.mean(e ** 2)), 2 * Xs.T @ e / len(ys)

## Task 1: Round to bfloat16

BF16 keeps FP32's 8 exponent bits and only 7 mantissa bits: it is the top 16 bits of an FP32 number. Round to nearest-even on the dropped 16 bits:
```
u = x.astype(np.float32).view(np.uint32)
u = u + 0x7FFF + ((u >> 16) & 1)       # round to nearest even
return (u & 0xFFFF0000).view(np.float32)
```
BF16 has FP32's range (up to ~3.4e38, tiny values survive) but coarse precision; FP16 has more precision but overflows above 65,504 and underflows below ~6e-8.

In [ ]:
def to_bf16(x):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_to_bf16(f):
    x = np.array([1.0, 1.00390625, 3.14159265, 70000.0, 1e-30], dtype=np.float32)
    y = f(x)
    assert y[0] == 1.0 and y[2] == np.float32(3.140625), f"precision: got {y[:3]}"
    assert y[3] == 70144.0, "70,000 is representable in BF16 (rounded), unlike FP16 which overflows"
    assert y[4] > 0, "1e-30 survives in BF16 (FP16 would flush it to 0)"
    assert np.isinf(np.float16(70000.0)) and np.float16(1e-30) == 0

check("to_bf16", _test_to_bf16, to_bf16)

## Task 2: A dynamic loss scaler

Multiply the loss by a scale S before backward so small gradients don't underflow in FP16; divide the gradients by S (in FP32) before the update.
* If any scaled gradient is inf or NaN: **skip** the step and halve S.
* After `interval` consecutive good steps: double S.

Implement `update(grads_fp16)` → returns `True` if the step should be applied, and adjusts `self.scale`.

In [ ]:
class LossScaler:
    def __init__(self, scale=2.0 ** 15, interval=50):
        self.scale, self.interval, self.good = scale, interval, 0

    def update(self, grads_fp16):
        # TODO
        raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_LossScaler(cls):
    s = cls(1024.0, interval=3)
    assert s.update(np.array([1.0], np.float16)) and s.scale == 1024.0
    assert not s.update(np.array([np.inf], np.float16)) and s.scale == 512.0, "overflow: skip and halve"
    for _ in range(3):
        assert s.update(np.array([1.0], np.float16))
    assert s.scale == 1024.0, "after `interval` good steps, double"

check("LossScaler", _test_LossScaler, LossScaler)

## Task 3: Train with FP16 gradients and FP32 master weights

Micikevicius et al. (2018): keep an FP32 **master copy** of the weights; compute gradients in half precision; scale the loss so they don't underflow.
For `steps` iterations: compute the FP32 gradient, emulate the half-precision backward as `np.float16(grad * scale)`, let the scaler decide, then (if applied) update the master weights with SGD using `grad16 / scale` (in FP32). With `use_scaling=False`, use scale 1 and never skip.
Return the final loss. The features are tiny, so unscaled FP16 gradients mostly flush to zero.

In [ ]:
def train_mixed(use_scaling, steps=300, lr=2e7, scaler_cls=None):
    w = np.zeros(Dm, dtype=np.float32)
    scaler = (scaler_cls or LossScaler)(2.0 ** 15, 50) if use_scaling else None
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_train_mixed(f):
    cls = LossScaler_solution if f.__name__.endswith('_solution') else LossScaler
    l0 = loss_and_grad(np.zeros(Dm))[0]
    ls = f(True, scaler_cls=cls); ln = f(False, scaler_cls=cls)
    ref = l0
    w = np.zeros(Dm)
    for _ in range(300):
        w = w - 2e7 * loss_and_grad(w)[1]
    ref = loss_and_grad(w)[0]
    assert ls < 1e-3 * l0, f"with loss scaling, loss should drop a lot ({l0:.2e} → {ls:.2e})"
    assert abs(ls - ref) < 0.05 * l0, "scaled FP16 training should track FP32 training"
    assert ln > 0.1 * l0, f"without scaling, gradients underflow and training stalls ({ln:.2e} of {l0:.2e})"

check("train_mixed", _test_train_mixed, train_mixed)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Round to bfloat16
def to_bf16_solution(x):
    u = np.asarray(x, dtype=np.float32).view(np.uint32).astype(np.uint64)
    u = (u + 0x7FFF + ((u >> 16) & 1)) & 0xFFFF0000
    return u.astype(np.uint32).view(np.float32)

check("to_bf16 (reference)", _test_to_bf16, to_bf16_solution)

In [ ]:
# @title Solution for Task 2: A dynamic loss scaler
class LossScaler_solution:
    def __init__(self, scale=2.0 ** 15, interval=50):
        self.scale, self.interval, self.good = scale, interval, 0

    def update(self, grads_fp16):
        if not np.all(np.isfinite(grads_fp16)):
            self.scale /= 2; self.good = 0
            return False
        self.good += 1
        if self.good >= self.interval:
            self.scale *= 2; self.good = 0
        return True

check("LossScaler (reference)", _test_LossScaler, LossScaler_solution)

In [ ]:
# @title Solution for Task 3: Train with FP16 gradients and FP32 master weights
def train_mixed_solution(use_scaling, steps=300, lr=2e7, scaler_cls=None):
    w = np.zeros(Dm, dtype=np.float32)
    scaler = (scaler_cls or LossScaler_solution)(2.0 ** 15, 50) if use_scaling else None
    for _ in range(steps):
        _, g = loss_and_grad(w)
        scale = scaler.scale if scaler else 1.0
        g16 = np.float16(g * scale)
        if scaler and not scaler.update(g16):
            continue
        w = (w - lr * (g16.astype(np.float32) / np.float32(scale))).astype(np.float32)
    return loss_and_grad(w)[0]

check("train_mixed (reference)", _test_train_mixed, train_mixed_solution)

---
## Experiments
**What fraction of gradients survives?** Cast the same gradient vector to FP16 with and without scaling, and to BF16. BF16's FP32-sized exponent is why most LLM training uses BF16 without loss scaling; FP8 training (DeepSeek-V3) needs fine-grained scales instead: one per 1×128 activation tile and per 128×128 weight block.

In [ ]:
np.seterr(over='ignore')
g = r.lognormal(-18, 2, 100000) * r.choice([-1, 1], 100000)
for name, conv in [('FP16, no scale', lambda v: np.float16(v)), ('FP16, ×2^15', lambda v: np.float16(v * 2 ** 15)), ('BF16, no scale', to_bf16_solution)]:
    print(f"{name:15} nonzero after cast: {np.mean(conv(g) != 0):6.1%}")

---
## Takeaways
* Mixed precision = low-precision math and storage, FP32 master weights and FP32 accumulation.
* FP16's narrow range needs dynamic loss scaling (skip and halve on overflow, grow when stable); BF16 has FP32's range, so it usually doesn't.
* Lower precision (FP8) needs finer-grained scaling; Llama 3 also accumulates and reduce-scatters gradients in FP32 for stability.